# Neural Networks

## A Neural Network in Pure JAX

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jrandom


def init_mlp(key, sizes):
    """Initialize parameters for an MLP."""
    params = []
    for in_size, out_size in zip(sizes[:-1], sizes[1:]):
        key, w_key, b_key = jrandom.split(key, 3)
        w = jrandom.normal(w_key, (out_size, in_size)) * jnp.sqrt(2.0 / in_size)
        b = jnp.zeros(out_size)
        params.append({"w": w, "b": b})
    return params

In [2]:
key = jrandom.PRNGKey(0)
key, init_key = jrandom.split(key)
params = init_mlp(init_key, [4, 16, 8, 2])

print("Number of layers:", len(params))
for i, layer in enumerate(params):
    print(f"  Layer {i}: w={layer['w'].shape}, b={layer['b'].shape}")

Number of layers: 3
  Layer 0: w=(16, 4), b=(16,)
  Layer 1: w=(8, 16), b=(8,)
  Layer 2: w=(2, 8), b=(2,)


## Forward Pass and Loss

In [3]:
def forward(params, x):
    """MLP forward pass."""
    for layer in params[:-1]:
        x = jnp.tanh(layer["w"] @ x + layer["b"])
    last = params[-1]
    return last["w"] @ x + last["b"]

In [4]:
def cross_entropy_loss(params, x, y_true):
    """Softmax cross-entropy loss for a single example."""
    logits = forward(params, x)
    log_probs = logits - jax.nn.logsumexp(logits)
    return -log_probs[y_true]

In [5]:
key, x_key = jrandom.split(key)
x = jrandom.normal(x_key, (4,))
y_true = 1

logits = forward(params, x)
print("Logits:", logits)
print("Loss:", cross_entropy_loss(params, x, y_true))

Logits: [ 0.78184485 -1.3003472 ]
Loss: 2.1996698


## Training Loop in Pure JAX

In [6]:
@jax.jit
def batch_loss(params, X, y):
    """Mean loss over a batch."""
    per_sample = jax.vmap(cross_entropy_loss, in_axes=(None, 0, 0))
    return jnp.mean(per_sample(params, X, y))


@jax.jit
def train_step(params, X, y, lr=0.01):
    loss, grads = jax.value_and_grad(batch_loss)(params, X, y)
    new_params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)
    return new_params, loss

In [7]:
data_key = jrandom.PRNGKey(1)
kX, ky = jrandom.split(data_key)
X_train = jrandom.normal(kX, (64, 4))
y_train = jrandom.randint(ky, (64,), 0, 2)

losses = []
for step in range(50):
    params, loss = train_step(params, X_train, y_train)
    if step % 10 == 0:
        losses.append(float(loss))
        print(f"Step {step:3d}: loss = {loss:.4f}")

Step   0: loss = 1.0776
Step  10: loss = 0.9893
Step  20: loss = 0.9162
Step  30: loss = 0.8572
Step  40: loss = 0.8104


## Pytree Operations: The Core Tool

In [8]:
params = init_mlp(jrandom.PRNGKey(0), [4, 8, 2])

scaled = jax.tree_util.tree_map(lambda p: p * 0.5, params)
print("Scaled w[0]:", scaled[0]["w"])

leaves = jax.tree_util.tree_leaves(params)
print("Number of parameter arrays:", len(leaves))
total_params = sum(l.size for l in leaves)
print("Total parameters:", total_params)

Scaled w[0]: [[-0.8635385  -0.71972173  0.07267086 -0.12499887]
 [-0.2693985  -0.41668096 -0.40595692  0.10506397]
 [-0.4633444   0.75314033 -0.06702395  0.34082976]
 [-0.46000835 -0.26470324 -0.13187484  0.15655014]
 [-0.42083442 -0.02448556 -0.33801785 -0.6925276 ]
 [-0.391028   -0.11384603  0.47034264  0.2871691 ]
 [-0.4097061  -0.1920648   0.28561118  0.584991  ]
 [-0.1447129   0.01905244 -0.22923982 -0.62493944]]
Number of parameter arrays: 4
Total parameters: 58


In [9]:
leaves, treedef = jax.tree_util.tree_flatten(params)
params_back = treedef.unflatten(leaves)

print("Treedef:", treedef)
print(
    "Round trip matches:",
    jax.tree_util.tree_all(
        jax.tree_util.tree_map(lambda a, b: (a == b).all(), params, params_back)
    ),
)

Treedef: PyTreeDef([{'b': *, 'w': *}, {'b': *, 'w': *}])
Round trip matches: True


## Introduction to Flax NNX

In [10]:
import flax.nnx as nnx


class MLP(nnx.Module):
    def __init__(self, in_features, hidden_features, out_features, rngs):
        self.layer1 = nnx.Linear(in_features, hidden_features, rngs=rngs)
        self.layer2 = nnx.Linear(hidden_features, out_features, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.layer1(x))
        return self.layer2(x)

In [11]:
# Initialize
model = MLP(4, 16, 2, rngs=nnx.Rngs(0))

# Call directly
x = jnp.ones(4)
print("Output:", model(x))
print("Output shape:", model(x).shape)

Output: [-0.19712375 -0.2312172 ]
Output shape: (2,)


## Training with Flax NNX and Optax

In [12]:
import optax

# Optimizer
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)


def loss_fn(model, X, y):
    logits = jax.vmap(model)(X)
    return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y))

In [13]:
@nnx.jit
def train_step(model, optimizer, X, y):
    loss, grads = nnx.value_and_grad(loss_fn)(model, X, y)
    optimizer.update(model, grads)
    return loss


# Training loop
losses = []
for step in range(50):
    loss = train_step(model, optimizer, X_train, y_train)
    if step % 10 == 0:
        losses.append(float(loss))
        print(f"Step {step:3d}: loss = {loss:.4f}")

Step   0: loss = 0.7473
Step  10: loss = 0.7167
Step  20: loss = 0.6929
Step  30: loss = 0.6754
Step  40: loss = 0.6632


## Choosing Between Approaches

In [14]:
# 1. Pure JAX: maximum visibility, manual everything
# Recommended for: understanding the fundamentals, simple models

# 2. Flax NNX: mutable modules, closest to PyTorch
# Recommended for: production ML, large research projects


print("Pure JAX parameters:", type(params))  # list of dicts
print("Flax model type:", type(model))  # MLP (nnx.Module subclass)

Pure JAX parameters: <class 'list'>
Flax model type: <class '__main__.MLP'>
